# HSQC-style offset plots: UCBShift N/H prediction vs ground truth

UCBShift was run separately on NMRBox; this notebook reads its output CSVs.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

UCBSHIFT_DIR = Path('../../../data/analysis/Fig1/random20_af2_structures/output')
GT_DIR = Path('../../../data/rBMRB/cluster_30p/NOREF')
FIGS_DIR = Path('Figs')          # separate folder so the 20 plots don't mix with the others
FIGS_DIR.mkdir(parents=True, exist_ok=True)

# the random 20 come from train / val / test -> merge the three NOREF splits, keyed by eid
gt = {'N': {}, 'H': {}, 'entry_ID': {}}
split_of = {}
for split in ['train', 'val', 'test']:
    g = json.load(open(GT_DIR / f'{split}.json'))
    for k, eid in g['entry_ID'].items():
        gt['N'][eid], gt['H'][eid], gt['entry_ID'][eid] = g['N'][k], g['H'][k], eid
        split_of[eid] = split

# entry_ID maps the key -> eid; here key == eid, so matched_NH works unchanged
eid2key = {v: k for k, v in gt['entry_ID'].items()}
print(f'{len(eid2key)} ground-truth entries available (train + val + test)')

csv_files = sorted(UCBSHIFT_DIR.glob('*.csv'))
missing = [f.stem for f in csv_files if f.stem not in eid2key]
if missing:
    raise KeyError(f'no ground truth for {missing}')
print(f'{len(csv_files)} UCBShift prediction files:',
      {s: [f.stem for f in csv_files if split_of[f.stem] == s] for s in ['train', 'val', 'test']})


In [ ]:
def matched_NH(csv_path):
    """For a UCBShift result CSV, return (res_idx, N_gt, H_gt, N_pred, H_pred)
    arrays for residues where BOTH the ground truth and the UCBShift prediction
    have non-null N and H shifts."""
    eid = csv_path.stem
    key = eid2key[eid]
    N_gt_all, H_gt_all = gt['N'][key], gt['H'][key]

    df = pd.read_csv(csv_path)
    n = min(len(df), len(N_gt_all), len(H_gt_all))

    res, n_gt, h_gt, n_pred, h_pred = [], [], [], [], []
    for i in range(n):
        row = df.iloc[i]
        ngt, hgt = N_gt_all[i], H_gt_all[i]
        npred, hpred = row['N_UCBShift'], row['H_UCBShift']
        if ngt is None or hgt is None or pd.isna(npred) or pd.isna(hpred):
            continue
        res.append(int(row['RESNUM']))
        n_gt.append(ngt); h_gt.append(hgt)
        n_pred.append(npred); h_pred.append(hpred)

    return (np.array(res), np.array(n_gt), np.array(h_gt),
            np.array(n_pred), np.array(h_pred))


In [ ]:
def set_style(base=12, family="Arial"):
    plt.rcParams.update({
        "font.family": family,
        "font.size": base,
        "axes.titlesize": base + 2,
        "axes.labelsize": base,
        "xtick.labelsize": base - 2,
        "ytick.labelsize": base - 2,
        "legend.fontsize": base - 2,
        "svg.fonttype": "none",
    })

set_style(base=16)

def plot_hsqc_offset(ax, csv_path, label=None):
    """HSQC-style plot: 1H (x) vs 15N (y), both axes inverted.
    Each matched residue is drawn for ground truth (H_gt, N_gt) and for the
    UCBShift prediction (H_pred, N_pred), connected by a line showing the
    peak displacement between the two."""
    eid = csv_path.stem
    res, n_gt, h_gt, n_pred, h_pred = matched_NH(csv_path)

    # ---- adjustable parameters ----
    LINE_COLOR = '#7f7f7f'   # connecting-line colour
    LINE_WIDTH = 0.8         # connecting-line width
    SIZE_GT    = 50          # marker size, ground truth (open circles)
    SIZE_PRED  = 50          # marker size, UCBShift prediction (filled dots)

    # connecting segments, single fixed color, no arrows
    segs = np.stack([np.column_stack([h_gt, n_gt]), np.column_stack([h_pred, n_pred])], axis=1)
    lc = LineCollection(segs, colors=LINE_COLOR, linewidths=LINE_WIDTH, alpha=0.5, zorder=2)
    ax.add_collection(lc)

    # peaks: ground truth vs UCBShift prediction
    ax.scatter(h_gt, n_gt, s=SIZE_GT, color='#E06C75', zorder=4,
               label='ground truth', alpha=0.8)
    ax.scatter(h_pred, n_pred, s=SIZE_PRED, color='#61AFEF', zorder=4,
               label='UCBShift', alpha=0.8)

    n_rmse = float(np.sqrt(np.mean((n_gt - n_pred) ** 2))) if len(res) else float('nan')

    ax.set_xlabel('$^{1}$H (ppm)')
    ax.set_ylabel('$^{15}$N (ppm)')
    # ax.set_xlim(9, 7); ax.set_ylim(130, 115)  # HSQC region
    ax.invert_xaxis(); ax.invert_yaxis()  # HSQC convention

    title = label or eid
    ax.set_title(f"{title} | $^{{15}}$N rmse = {n_rmse:.3f} | n = {len(res)}")
    ax.legend(loc='best', framealpha=0.9)
    # ax.grid(True, ls=':', alpha=0.4)


In [ ]:
def n_rmse_of(csv_path):
    _, n_gt, _, n_pred, _ = matched_NH(csv_path)
    return float(np.sqrt(np.mean((n_gt - n_pred) ** 2))) if len(n_gt) else np.inf


rmse = {f: n_rmse_of(f) for f in csv_files}
csv_sorted = sorted(csv_files, key=rmse.get)          # lowest 15N RMSE first; add reverse=True for worst first
print(pd.DataFrame({'eid': [f.stem for f in csv_sorted],
                    'N_rmse': [round(rmse[f], 3) for f in csv_sorted]}).to_string(index=False))

for rank, csv_path in enumerate(csv_sorted, start=1):
    eid = csv_path.stem
    fig, ax = plt.subplots(figsize=(7, 6))
    plot_hsqc_offset(ax, csv_path, label=eid)
    fig.tight_layout()
    stem = f'hsqc_offset_{rank:02d}_{eid}'                # rank prefix -> files sort by RMSE in the folder
    fig.savefig(FIGS_DIR / f'{stem}.svg', format='svg')
    fig.savefig(FIGS_DIR / f'{stem}.png', format='png', dpi=300)
    plt.show()
